# Aligment of entities

### Load libraries and define functions


In [ ]:
import requests
import os
import re

API_URL = "https://aymurai.collectiveai.io"
DATA_DIR = '/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/'
doc_path = "aymurai - ejemplo 02.docx"

def normalize_text(text: str) -> str:
    """Normalize text spacing to avoid API issues"""
    # Replace single space after period with double space before Roman numerals
    text = re.sub(r'\.(\s+)([IVX]+\.)', r'.  \2', text)
    # Clean up any multiple spaces
    text = re.sub(r'\s+', ' ', text)
    return text


In [ ]:
# Celda 1: Importar y definir el alineador mejorado
import difflib
import re
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from enum import Enum

class AlignmentStatus(Enum):
    MATCH_EXACT = "match_exact"
    MATCH_FUZZY = "match_fuzzy"
    MATCH_LESSER = "match_lesser"
    NO_MATCH = None

@dataclass
class CharInterval:
    start_pos: int
    end_pos: int

@dataclass
class Extraction:
    extraction_class: str
    extraction_text: str
    char_interval: CharInterval
    alignment_status: AlignmentStatus
    extraction_index: int
    group_index: int
    description: Optional[str] = None
    attributes: Dict[str, Any] = None

    def __post_init__(self):
        if self.attributes is None:
            self.attributes = {}

@dataclass
class AnnotatedDocument:
    document_id: str
    text: str
    extractions: List[Extraction]

class ImprovedEntityAligner:
    def __init__(self, fuzzy_threshold: float = 0.75):
        self.fuzzy_threshold = fuzzy_threshold
    
    def _normalize_text(self, text: str) -> str:
        """Normalizar texto para comparación - más flexible"""
        normalized = re.sub(r'\s+', ' ', text.strip().lower())
        normalized = re.sub(r'[.,;:!?]+$', '', normalized)
        return normalized
    
    def _texts_match(self, text1: str, text2: str) -> bool:
        """Verificar si dos textos coinciden (flexible con mayúsculas/puntuación)"""
        return self._normalize_text(text1) == self._normalize_text(text2)
    
    def _find_entity_in_text(self, entity_text: str, source_text: str, 
                            start_search: int = 0) -> Optional[Dict[str, Any]]:
        entity_normalized = self._normalize_text(entity_text)
        
        # Match exacto
        exact_pos = source_text.find(entity_text, start_search)
        if exact_pos != -1:
            return {
                'start_char': exact_pos,
                'end_char': exact_pos + len(entity_text),
                'status': AlignmentStatus.MATCH_EXACT,
                'score': 1.0
            }
        
        # Match exacto normalizado
        for i in range(start_search, len(source_text) - len(entity_text) + 1):
            window_text = source_text[i:i + len(entity_text)]
            if self._texts_match(window_text, entity_text):
                return {
                    'start_char': i,
                    'end_char': i + len(entity_text),
                    'status': AlignmentStatus.MATCH_EXACT,
                    'score': 1.0
                }
        
        # Match con ventana flexible
        for window_size in range(len(entity_text) - 2, len(entity_text) + 3):
            if window_size <= 0 or window_size > len(source_text):
                continue
                
            for i in range(start_search, len(source_text) - window_size + 1):
                window_text = source_text[i:i + window_size]
                if self._texts_match(window_text, entity_text):
                    return {
                        'start_char': i,
                        'end_char': i + window_size,
                        'status': AlignmentStatus.MATCH_EXACT,
                        'score': 1.0
                    }
        
        # Match fuzzy
        window_size = len(entity_text)
        best_match = None
        best_score = 0.0
        
        for i in range(start_search, len(source_text) - window_size + 1):
            window_text = source_text[i:i + window_size]
            window_normalized = self._normalize_text(window_text)
            
            matcher = difflib.SequenceMatcher(None, entity_normalized, window_normalized)
            ratio = matcher.ratio()
            
            if ratio >= self.fuzzy_threshold and ratio > best_score:
                best_score = ratio
                best_match = {
                    'start_char': i,
                    'end_char': i + window_size,
                    'status': AlignmentStatus.MATCH_FUZZY,
                    'score': ratio
                }
        
        return best_match
    
    def _find_all_occurrences(self, entity_text: str, source_text: str) -> List[Dict[str, Any]]:
        occurrences = []
        start_search = 0
        
        while True:
            match = self._find_entity_in_text(entity_text, source_text, start_search)
            if not match:
                break
            
            occurrences.append(match)
            start_search = match['end_char']
        
        return occurrences
    
    def align_entities_blind(self, entities: List[Dict[str, Any]], 
                           source_text: str) -> AnnotatedDocument:
        extractions = []
        extraction_index = 1
        
        for entity in entities:
            entity_text = entity['text']
            entity_class = entity.get('attrs', {}).get('aymurai_label', '')
            entity_attrs = entity.get('attrs', {})
            
            occurrences = self._find_all_occurrences(entity_text, source_text)
            
            if not occurrences:
                extraction = Extraction(
                    extraction_class=entity_class,
                    extraction_text=entity_text,
                    char_interval=CharInterval(start_pos=0, end_pos=0),
                    alignment_status=AlignmentStatus.NO_MATCH,
                    extraction_index=extraction_index,
                    group_index=0,
                    description=None,
                    attributes=entity_attrs
                )
                extractions.append(extraction)
                extraction_index += 1
            else:
                for i, occurrence in enumerate(occurrences):
                    extraction = Extraction(
                        extraction_class=entity_class,
                        extraction_text=entity_text,
                        char_interval=CharInterval(
                            start_pos=occurrence['start_char'],
                            end_pos=occurrence['end_char']
                        ),
                        alignment_status=occurrence['status'],
                        extraction_index=extraction_index,
                        group_index=i,
                        description=None,
                        attributes=entity_attrs
                    )
                    extractions.append(extraction)
                    extraction_index += 1
        
        return AnnotatedDocument(
            document_id="legal_document_001",
            text=source_text,
            extractions=extractions
        )

In [ ]:
# Function to extract document using the API
def extract_document(file_path: str) -> dict:
    # Open the file in binary mode and send the POST request
    with open(file_path, "rb") as file:
        files = {"file": file}
        response = requests.post(url=f"{API_URL}/document-extract", files=files)
    response.raise_for_status()
    return response.json()

# The anonymized document information including the text and labels.
def predict_anonymization(text: str) -> dict:
    data = {"text": text}
    response = requests.post(
        url=f"{API_URL}/anonymizer/predict?use_cache=true&echo=false", 
        json=data,  
        headers={"Content-Type": "application/json"}
    )
    response.raise_for_status()
    return response.json()

# Extract and normalize the document content
extracted_document = extract_document(os.path.join(DATA_DIR, doc_path))
document = " ".join(extracted_document["document"])

# Normalize the text to fix spacing issues
normalized_document = normalize_text(document)



### Prepare NER output

In [ ]:
# Extract and print the document content
extracted_document = extract_document(os.path.join(DATA_DIR, doc_path))
document = " ".join(extracted_document["document"]).replace('\n', '').replace('"', '').strip()
#predict_anonymization(document)

In [ ]:
docum = 'JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar SENTENCIA En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar, Expte. N.o 3187/2023.  I. ANTECEDENTES Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial. La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó a aislarla de su entorno familiar, controlar sus gastos y proferir insultos y amenazas, que en los últimos meses derivaron en empujones y golpes. Se acompañaron constancias médicas del Hospital San Martín de fechas 14/08/2023 y 08/11/2023, donde se registran lesiones en antebrazos y región lumbar, así como un informe psicológico que describe un cuadro de depresión moderada y ansiedad. II. MEDIDAS CAUTELARES ADOPTADAS Con fecha 11 de noviembre de 2023, este Juzgado dispuso: Exclusión inmediata del denunciado del domicilio conyugal. Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros. Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales. III. PRUEBA PRODUCIDA En audiencia celebrada el 17 de noviembre de 2023, declararon la Sra. Patricia Gómez, vecina del domicilio conyugal, y el Sr. Luis Alberto Rivas, compañero de trabajo de la denunciante, quienes relataron haber presenciado episodios de gritos, discusiones y amenazas por parte del denunciado. El Equipo Interdisciplinario del Juzgado emitió informe en el que concluyó que existe un riesgo alto de reiteración de la violencia, con impacto negativo en el bienestar emocional de la menor. IV. FUNDAMENTOS De la valoración integral de la prueba surge acreditada la existencia de violencia física, psicológica y patrimonial ejercida por el Sr. Diego Esteban Fernández contra la Sra. Ana Carolina Rodríguez, en el marco de una relación de pareja y con afectación a la hija menor. La Ley Nacional 26.485 y la Ley Provincial 11.529 obligan a adoptar medidas urgentes y eficaces para proteger a las víctimas de violencia de género. En este caso, la reiteración de los hechos, la proximidad de los domicilios y la vulnerabilidad de la menor justifican la extensión de las medidas cautelares y la adopción de acciones complementarias. V. RESUELVO Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez y de la hija menor M.F.R. Mantener la exclusión del denunciado del domicilio conyugal. Ordenar al denunciado la asistencia obligatoria a un programa de tratamiento para agresores, debiendo acreditar su cumplimiento ante este Juzgado. Disponer que el Equipo Interdisciplinario realice seguimiento quincenal del grupo familiar y remita informes a este Tribunal. Notifíquese a las partes, al Ministerio Público y a la Comisaría de la Mujer de San Lorenzo. Regístrese, notifíquese y archívese. Fdo.: Dra. Verónica Salvatierra – Jueza de Familia N.o 1 – San Lorenzo'
texto = "JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar SENTENCIA En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar, Expte. N.o 3187/2023.  I. ANTECEDENTES Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial. La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó a aislarla de su entorno familiar, controlar sus gastos y proferir insultos y amenazas, que en los últimos meses derivaron en empujones y golpes. Se acompañaron constancias médicas del Hospital San Martín de fechas 14/08/2023 y 08/11/2023, donde se registran lesiones en antebrazos y región lumbar, así como un informe psicológico que describe un cuadro de depresión moderada y ansiedad. II. MEDIDAS CAUTELARES ADOPTADAS Con fecha 11 de noviembre de 2023, este Juzgado dispuso: Exclusión inmediata del denunciado del domicilio conyugal. Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros. Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales. III. PRUEBA PRODUCIDA En audiencia celebrada el 17 de noviembre de 2023, declararon la Sra. Patricia Gómez, vecina del domicilio conyugal, y el Sr. Luis Alberto Rivas, compañero de trabajo de la denunciante, quienes relataron haber presenciado episodios de gritos, discusiones y amenazas por parte del denunciado. El Equipo Interdisciplinario del Juzgado emitió informe en el que concluyó que existe un riesgo alto de reiteración de la violencia, con impacto negativo en el bienestar emocional de la menor. IV. FUNDAMENTOS De la valoración integral de la prueba surge acreditada la existencia de violencia física, psicológica y patrimonial ejercida por el Sr. Diego Esteban Fernández contra la Sra. Ana Carolina Rodríguez, en el marco de una relación de pareja y con afectación a la hija menor. La Ley Nacional 26.485 y la Ley Provincial 11.529 obligan a adoptar medidas urgentes y eficaces para proteger a las víctimas de violencia de género. En este caso, la reiteración de los hechos, la proximidad de los domicilios y la vulnerabilidad de la menor justifican la extensión de las medidas cautelares y la adopción de acciones complementarias. V. RESUELVO Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez y de la hija menor M.F.R. Mantener la exclusión del denunciado del domicilio conyugal. Ordenar al denunciado la asistencia obligatoria a un programa de tratamiento para agresores, debiendo acreditar su cumplimiento ante este Juzgado. Disponer que el Equipo Interdisciplinario realice seguimiento quincenal del grupo familiar y remita informes a este Tribunal. Notifíquese a las partes, al Ministerio Público y a la Comisaría de la Mujer de San Lorenzo. Regístrese, notifíquese y archívese. Fdo.: Dra. Verónica Salvatierra – Jueza de Familia N.o 1 – San Lorenzo"
text =  "JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar SENTENCIA En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar, Expte. N.o 3187/2023.  I. ANTECEDENTES Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial. La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó a aislarla de su entorno familiar, controlar sus gastos y proferir insultos y amenazas, que en los últimos meses derivaron en empujones y golpes. Se acompañaron constancias médicas del Hospital San Martín de fechas 14/08/2023 y 08/11/2023, donde se registran lesiones en antebrazos y región lumbar, así como un informe psicológico que describe un cuadro de depresión moderada y ansiedad. II. MEDIDAS CAUTELARES ADOPTADAS Con fecha 11 de noviembre de 2023, este Juzgado dispuso: Exclusión inmediata del denunciado del domicilio conyugal. Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros. Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales. III. PRUEBA PRODUCIDA En audiencia celebrada el 17 de noviembre de 2023, declararon la Sra. Patricia Gómez, vecina del domicilio conyugal, y el Sr. Luis Alberto Rivas, compañero de trabajo de la denunciante, quienes relataron haber presenciado episodios de gritos, discusiones y amenazas por parte del denunciado. El Equipo Interdisciplinario del Juzgado emitió informe en el que concluyó que existe un riesgo alto de reiteración de la violencia, con impacto negativo en el bienestar emocional de la menor. IV. FUNDAMENTOS De la valoración integral de la prueba surge acreditada la existencia de violencia física, psicológica y patrimonial ejercida por el Sr. Diego Esteban Fernández contra la Sra. Ana Carolina Rodríguez, en el marco de una relación de pareja y con afectación a la hija menor. La Ley Nacional 26.485 y la Ley Provincial 11.529 obligan a adoptar medidas urgentes y eficaces para proteger a las víctimas de violencia de género. En este caso, la reiteración de los hechos, la proximidad de los domicilios y la vulnerabilidad de la menor justifican la extensión de las medidas cautelares y la adopción de acciones complementarias. V. RESUELVO Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez y de la hija menor M.F.R. Mantener la exclusión del denunciado del domicilio conyugal. Ordenar al denunciado la asistencia obligatoria a un programa de tratamiento para agresores, debiendo acreditar su cumplimiento ante este Juzgado. Disponer que el Equipo Interdisciplinario realice seguimiento quincenal del grupo familiar y remita informes a este Tribunal. Notifíquese a las partes, al Ministerio Público y a la Comisaría de la Mujer de San Lorenzo. Regístrese, notifíquese y archívese. Fdo.: Dra. Verónica Salvatierra – Jueza de Familia N.o 1 – San Lorenzo"


In [ ]:
from pprint import pprint
text =  "JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar SENTENCIA En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar, Expte. N.o 3187/2023.  I. ANTECEDENTES Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial. La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó a aislarla de su entorno familiar, controlar sus gastos y proferir insultos y amenazas, que en los últimos meses derivaron en empujones y golpes. Se acompañaron constancias médicas del Hospital San Martín de fechas 14/08/2023 y 08/11/2023, donde se registran lesiones en antebrazos y región lumbar, así como un informe psicológico que describe un cuadro de depresión moderada y ansiedad. II. MEDIDAS CAUTELARES ADOPTADAS Con fecha 11 de noviembre de 2023, este Juzgado dispuso: Exclusión inmediata del denunciado del domicilio conyugal. Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros. Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales. III. PRUEBA PRODUCIDA En audiencia celebrada el 17 de noviembre de 2023, declararon la Sra. Patricia Gómez, vecina del domicilio conyugal, y el Sr. Luis Alberto Rivas, compañero de trabajo de la denunciante, quienes relataron haber presenciado episodios de gritos, discusiones y amenazas por parte del denunciado. El Equipo Interdisciplinario del Juzgado emitió informe en el que concluyó que existe un riesgo alto de reiteración de la violencia, con impacto negativo en el bienestar emocional de la menor. IV. FUNDAMENTOS De la valoración integral de la prueba surge acreditada la existencia de violencia física, psicológica y patrimonial ejercida por el Sr. Diego Esteban Fernández contra la Sra. Ana Carolina Rodríguez, en el marco de una relación de pareja y con afectación a la hija menor. La Ley Nacional 26.485 y la Ley Provincial 11.529 obligan a adoptar medidas urgentes y eficaces para proteger a las víctimas de violencia de género. En este caso, la reiteración de los hechos, la proximidad de los domicilios y la vulnerabilidad de la menor justifican la extensión de las medidas cautelares y la adopción de acciones complementarias. V. RESUELVO Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez y de la hija menor M.F.R. Mantener la exclusión del denunciado del domicilio conyugal. Ordenar al denunciado la asistencia obligatoria a un programa de tratamiento para agresores, debiendo acreditar su cumplimiento ante este Juzgado. Disponer que el Equipo Interdisciplinario realice seguimiento quincenal del grupo familiar y remita informes a este Tribunal. Notifíquese a las partes, al Ministerio Público y a la Comisaría de la Mujer de San Lorenzo. Regístrese, notifíquese y archívese. Fdo.: Dra. Verónica Salvatierra – Jueza de Familia N.o 1 – San Lorenzo"
ner_predictions = predict_anonymization(docum)

ner_predictions


In [ ]:
# entities as list of {'text': ..., 'label': ...} from ner_predictions['labels']
entities = [
    {'text': ent['text'],'attrs':{ 'label': ent['attrs']['aymurai_label']}}
    for ent in ner_predictions.get('labels', [])
]
doc = ner_predictions['document']
doc_entities = {'document': doc, 'labels': entities}
pprint(doc_entities)

In [ ]:
entities

## Version 1

In [ ]:
from entity_aligner_minimal import align_ner_data, create_langextract_alignment, validate_alignment

# Crear alineación en formato langextract
annotated_doc, stats = align_ner_data(doc_entities, fuzzy_threshold=0.75)

print(f"AnnotatedDocument creado:")
print(f"  Total extracciones: {len(annotated_doc.extractions)}")
print(f"  Precisión: {stats['alignment_accuracy']:.2%}")
print(f"  Matches exactos: {stats['exact_matches']}")
print(f"  Matches fuzzy: {stats['fuzzy_matches']}")
print(f"  Sin matches: {stats['no_matches']}")

In [ ]:
# Mostrar algunas extracciones
for i, ext in enumerate(annotated_doc.extractions[:10]):
    print(f"{i+1}. {ext.extraction_class}: '{ext.extraction_text}'")
    print(f"   Posición: {ext.char_interval.start_pos}-{ext.char_interval.end_pos}")
    print(f"   Estado: {ext.alignment_status.value}")
    print(f"   Grupo: {ext.group_index}")
    print()

# Filtrar por tipo
personas = [ext for ext in annotated_doc.extractions if ext.extraction_class == 'PER']
fechas = [ext for ext in annotated_doc.extractions if ext.extraction_class == 'FECHA']
ubicaciones = [ext for ext in annotated_doc.extractions if ext.extraction_class == 'LOC']

print(f"Personas: {len(personas)}")
print(f"Fechas: {len(fechas)}")
print(f"Ubicaciones: {len(ubicaciones)}")

In [ ]:
doc[581:592]

In [ ]:
from rich.pretty import pprint

pprint(annotated_doc.extractions)

## Version 2

### Usar alineador

In [ ]:
# Crear el alineador
aligner = BlindEntityAligner(fuzzy_threshold=0.75)

# Extraer texto y entidades de tu variable
source_text = ner_predictions['document']
entities = ner_predictions['labels']

# Alinear entidades
annotated_doc = aligner.align_entities_blind(entities, source_text)

print(f"Documento procesado: {len(annotated_doc.text)} caracteres")
print(f"Total extracciones: {len(annotated_doc.extractions)}")

In [ ]:
pprint(annotated_doc)

In [ ]:
pprint(annotated_doc.extractions[0])

### Print resultados

In [ ]:
# Mostrar algunas extracciones
for i, extraction in enumerate(annotated_doc.extractions[:10]):
    print(f"{i+1}. {extraction.extraction_class}: '{extraction.extraction_text}'")
    print(f"   Posición: {extraction.char_interval.start_pos}-{extraction.char_interval.end_pos}")
    print(f"   Estado: {extraction.alignment_status.value}")
    print(f"   Grupo: {extraction.group_index}")
    print()

# Estadísticas
total = len(annotated_doc.extractions)
exact_matches = sum(1 for ext in annotated_doc.extractions 
                   if ext.alignment_status == AlignmentStatus.MATCH_EXACT)
fuzzy_matches = sum(1 for ext in annotated_doc.extractions 
                   if ext.alignment_status == AlignmentStatus.MATCH_FUZZY)
no_matches = sum(1 for ext in annotated_doc.extractions 
                if ext.alignment_status == AlignmentStatus.NO_MATCH)

print("Estadísticas:")
print(f"  Total extracciones: {total}")
print(f"  Matches exactos: {exact_matches}")
print(f"  Matches fuzzy: {fuzzy_matches}")
print(f"  Sin matches: {no_matches}")
print(f"  Precisión: {exact_matches/total:.2%}")

### Validar posiciones

In [ ]:
# Validar que las posiciones son correctas
valid_positions = 0
for extraction in annotated_doc.extractions:
    start = extraction.char_interval.start_pos
    end = extraction.char_interval.end_pos
    if (0 <= start < end <= len(annotated_doc.text)):
        actual_text = annotated_doc.text[start:end]
        if actual_text.strip() == extraction.extraction_text.strip():
            valid_positions += 1

print(f"Posiciones válidas: {valid_positions}/{total} ({valid_positions/total:.2%})")

# Mostrar algunas validaciones
print("\nValidaciones de posiciones:")
for i, extraction in enumerate(annotated_doc.extractions[:5]):
    start = extraction.char_interval.start_pos
    end = extraction.char_interval.end_pos
    actual_text = annotated_doc.text[start:end]
    is_correct = actual_text.strip() == extraction.extraction_text.strip()
    status = "✓" if is_correct else "✗"
    print(f"  {i+1}. '{extraction.extraction_text}' -> '{actual_text}' {status}")

In [ ]:
# Acceder a las extracciones alineadas
aligned_extractions = annotated_doc.extractions

# Filtrar por tipo de entidad
personas = [ext for ext in aligned_extractions if ext.extraction_class == 'PER']
fechas = [ext for ext in aligned_extractions if ext.extraction_class == 'FECHA']
ubicaciones = [ext for ext in aligned_extractions if ext.extraction_class == 'LOC']

print(f"Personas encontradas: {len(personas)}")
print(f"Fechas encontradas: {len(fechas)}")
print(f"Ubicaciones encontradas: {len(ubicaciones)}")

# Mostrar personas únicas
personas_unicas = list(set(ext.extraction_text for ext in personas))
print(f"\nPersonas únicas: {personas_unicas}")